# Lab 3 · Handling API errors
**After Demo 2 · ~15 minutes · needs internet for the live tests**

Your task: write **`safe_get_json()`**, a function you'll reuse in the dashboard lab.
It must never crash. It returns a tuple `(data, error)`:
- success → `(parsed_json, None)`
- failure → `(None, "a short, human-friendly message")`

> 🧪 Cells marked **CHECK** test your answer. Run them after each exercise: they print ✅ or raise an error with a hint.

In [ ]:
import time
import requests

## Exercise 3.1 · Write the function
Requirements:
1. Use `timeout=timeout` and call `raise_for_status()`
2. Handle separately: **Timeout**, **HTTPError**, **ConnectionError**, **invalid JSON** (`ValueError`)
3. **Retry** up to `retries` extra times for timeouts, connection errors and 5xx errors, waiting `0.5 * attempt` seconds between tries
4. **Don't retry** 4xx errors or invalid JSON (retrying won't fix them)

Message ideas: `"Timed out after 2s"`, `"HTTP 404: Not Found"`, `"Could not connect"`, `"Response was not JSON"`.

In [ ]:
def safe_get_json(url, params=None, timeout=5, retries=2):
    error = None
    for attempt in range(1, retries + 2):
        try:
            ...
        except requests.exceptions.Timeout:
            ...
        except requests.exceptions.HTTPError as e:
            ...
        except requests.exceptions.ConnectionError:
            ...
        except ValueError:
            ...
        time.sleep(0.5 * attempt)
    return None, error

## Exercise 3.2 · Test it against every failure
Run the cell below. Every row should show either data or a clear message — and **no traceback**.

In [ ]:
cases = [
    ("Good request",   "https://jsonplaceholder.typicode.com/users/1", 5),
    ("404 Not Found",  "https://jsonplaceholder.typicode.com/does-not-exist", 5),
    ("401 Unauthorized", "https://httpbin.org/status/401", 5),
    ("503 Unavailable","https://httpbin.org/status/503", 5),
    ("Too slow",       "https://httpbin.org/delay/3", 1),
    ("Bad host",       "https://api.this-host-does-not-exist.example-invalid", 5),
    ("Not JSON",       "https://httpbin.org/html", 5),
]
results = []
for label, url, t in cases:
    start = time.perf_counter()
    data, error = safe_get_json(url, timeout=t, retries=1)
    results.append((label, "OK" if data is not None else error, round(time.perf_counter() - start, 1)))

for label, outcome, secs in results:
    print(f"{label:17} → {outcome:28} ({secs}s)")

In [ ]:
# CHECK 3.2
outcome = {label: out for label, out, _ in results}
assert outcome["Good request"] == "OK"
assert "404" in outcome["404 Not Found"]
assert "503" in outcome["503 Unavailable"]
assert "Timed out" in outcome["Too slow"] or "timed out" in outcome["Too slow"].lower()
assert outcome["Bad host"] is not None and outcome["Bad host"] != "OK"
assert "JSON" in outcome["Not JSON"]
print("✅ 3.2 correct — your function survives every failure")

## Exercise 3.3 · Use it for real
Fetch the **current temperature** for every city in `CITIES` with `safe_get_json`.
If a call fails, print a warning and continue with the next city. Finish with a dict `temps = {city: temp}`.

One city has deliberately **bad coordinates** (latitude 200) so you can see your handling work.

In [ ]:
URL = "https://api.open-meteo.com/v1/forecast"
CITIES = {"Delhi": (28.61, 77.21), "Mumbai": (19.08, 72.88),
          "Nowhere": (200, 0),            # invalid on purpose
          "Kolkata": (22.57, 88.36)}

temps = {}
for city, (lat, lon) in CITIES.items():
    ...
temps

## ⭐ Bonus
Some APIs answer `429 Too Many Requests` with a `Retry-After` header (seconds to wait).
Extend your function: on 429, read `e.response.headers.get("Retry-After")`, wait that long (max 5 s) and retry.